# M01 · Python for earthquake catalogs

**Week 1 · about 10 h (+ 1 h optional)** · before this: M00

> **한국어 요약** 이 모듈에서는 지진 목록(지진 카탈로그)을 다루는 데 필요한 파이썬 기초를 익힙니다. 변수, 리스트, 딕셔너리, 반복문, 독스트링이 있는 함수, 오류 메시지(트레이스백) 읽는 법을 배운 뒤, NumPy 배열로 여러 지점 쌍의 대원 거리를 한 번에 계산하는 함수를 직접 작성합니다. pandas로 CSV 지진 목록을 읽어 시각을 UTC로 맞추고, 비어 있는 깊이를 NaN으로 처리하며, 조건으로 고르고 시간 순서로 정렬합니다. 마지막으로, 같은 UTC 시각을 KST로 바꾸는 두 가지 방법(언제나 9시간을 더하는 고정 차이와 'Asia/Seoul' 시간대)을 비교하는 함수를 작성하여, 오래된 기록에서 한 시간이 어긋나는 사건을 직접 찾아내고 그 까닭을 설명합니다.

**You will**

- Write a documented, vectorized NumPy function (great-circle distance) that agrees with an independent library to one part in a million
- Convert KST clock readings to UTC with a fixed 9-hour offset across midnight, month, year and leap-day changes
- Load a catalog CSV into a pandas table with time-zone-aware UTC times, NaN for blank depths and columns chosen by name
- Select the events within a radius and sort them by origin time with boolean filtering and sort_values
- Write a function that compares two ways of turning UTC times into KST (a fixed 9-hour offset and the 'Asia/Seoul' time zone), then find, count and explain the events where they disagree, reading tracebacks from the bottom up along the way

**New words:** prediction (예측), random seed (난수 시드), earthquake catalog (지진 목록, 지진 카탈로그), mainshock (본진), epicenter (진앙), hypocenter (진원), focal depth (진원 깊이), local magnitude (ML; 국지 규모) · [all terms](../../../docs/glossary.md)

**Optional reading:** Programming with Python, the lessons on lists, loops, functions and errors; Plotting and Programming in Python, the lessons on reading tabular data and pandas DataFrames

Log any AI help in `work/ai_log/M01.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M01.yaml](../../../.sgtl/criteria/M01.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
EXAMPLE_LAT = 35.76        # degrees north: the 2016 Gyeongju epicenter area (Part 4)
EXAMPLE_LON = 129.19       # degrees east
EXAMPLE_RADIUS_KM = 15.0   # search radius in kilometers (Part 4)
MIN_ML = 3.0               # smallest local magnitude in the summary tables
N_SHOW = 5                 # how many rows head() and tail() show
DEMO_SEED = 2016           # random seed of the speed demo (Part 2)
N_DEMO = 1_000_000         # number of values in the speed demo (Part 2)

In [ ]:
import sgtl_tutorial as st
S = st.init("M01")

import datetime as dt
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import Image, display


def show_figure(figure, filename):
    """Save and display the same static figure, including in an archived notebook."""
    path = Path("figures") / filename
    path.parent.mkdir(exist_ok=True)
    figure.savefig(path, dpi=150, bbox_inches="tight")
    display(Image(filename=str(path)))
    plt.close(figure)

## How this notebook works

Every task has the same five steps:

1. Read the task. Its **Contract** says exactly what goes in and what comes out, with units.
2. Write a **prediction** (glossary `prediction`): a short written guess of the result, made before you run
   the code, so you can learn from the difference.
3. Write your code: replace the line `raise NotImplementedError(...)` under `# YOUR CODE HERE`.
4. Run the check cell. It prints PASS or FAIL, and on FAIL a short reason.
5. Answer the **Explain** prompt in 3-5 sentences of your own.

The checks of tasks 1-4 try your function on fresh practice cases drawn at random. They come from a
**random seed** (glossary `random_seed`): a number that fixes a random-number generator, so the same "random"
values come out every time. Your seed is made from your user name and the task; `st.reroll("M01-T1")` draws
new cases. `st.hint("M01-T1", 1)` shows a first hint; hints 2 and 3 unlock after failed checks.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M01.md`. Work through the shared examples and core tasks in order. A kernel restart clears their
variables and function definitions: rerun the parameters, setup and completed shared/core cells in order.
Full **Run All Cells** also runs optional tasks; core submission uses `bin/sgtl submit M01 --apply` and includes
optional work only with `--include-optional`.

## Part 1 · Python in context (about 90 min)

A **variable** is a name for a value. Every value has a **type**: `str` (text), `int` (whole number),
`float` (decimal number) or `bool` (`True` or `False`). Below, one row of an **earthquake catalog**
(glossary `earthquake_catalog`: a table of earthquakes with origin time, location, depth and magnitude, made
by a seismological agency) is kept in a few variables: the 2016 Gyeongju **mainshock** (glossary `mainshock`:
the largest earthquake of a sequence) as listed in the KMA catalog that comes with this course.

In [ ]:
event_id = "2016000291"   # str: an ID is a label, so keep it as text
latitude = 35.76          # float, degrees north
longitude = 129.19        # float, degrees east
depth_km = 15.0           # float, kilometers
ml = 5.8                  # float, local magnitude
year = 2016               # int
print(type(event_id), type(ml), type(year))
print(int("0042"), "versus", "0042")   # a number forgets leading zeros; text keeps them

The **epicenter** (glossary `epicenter`) is the point on the Earth's surface directly above the
**hypocenter** (glossary `hypocenter`), the point inside the Earth where the rupture starts. The
**focal depth** (glossary `focal_depth`) is the depth of the hypocenter below the surface, in kilometers.
The **local magnitude** ML (glossary `local_magnitude`) comes from the largest amplitudes on nearby
seismometers, corrected for distance. Any glossary entry can be printed:

In [ ]:
print(st.glossary.term("hypocenter"))
print(st.glossary.term("local_magnitude"))

**Read the same earthquake in two views.** The star on the surface is the epicenter; the star below it is
the hypocenter. The dashed vertical line connects them. Depth is a distance below the surface, whereas
latitude and longitude locate a position on the surface. This schematic uses the example variables above;
it is not a cross-section through a measured fault. Run this provided plotting cell; M02 teaches figure building.

**Look:** which coordinate changes if a location estimate moves deeper without moving its epicenter?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
axes[0].plot(longitude, latitude, "*", color="crimson", ms=17)
axes[0].annotate("Epicenter", (longitude, latitude), xytext=(12, 14), textcoords="offset points")
axes[0].set(xlim=(longitude - 0.15, longitude + 0.15), ylim=(latitude - 0.15, latitude + 0.15),
            xlabel="Longitude (°E)", ylabel="Latitude (°N)", title="Surface position: map view")
axes[1].axhline(0, color="0.2", lw=2)
axes[1].plot([0, 0], [0, depth_km], "--", color="0.5")
axes[1].plot(0, 0, "*", color="crimson", ms=14)
axes[1].plot(0, depth_km, "*", color="tab:blue", ms=14)
axes[1].annotate("Hypocenter", (0, depth_km), xytext=(12, 0), textcoords="offset points")
axes[1].set(xlim=(-10, 10), ylim=(22, -2), xlabel="Horizontal offset from epicenter (km)",
            ylabel="Depth below surface (km)", title="Source position: schematic section")
for ax in axes:
    ax.grid(alpha=0.2)
show_figure(fig, "M01_epicenter_hypocenter.png")

A **list** keeps values in order. Positions start at 0, negative positions count from the end, and a
slice `a[i:j]` takes positions `i` up to (not including) `j`. A **dictionary** (`dict`) maps keys to values,
like one row of a table with named fields.

In [ ]:
mags = [2.3, 5.1, 3.0, 5.8, 2.7]   # practice numbers
print(mags[0], mags[-1], len(mags), mags[1:3])
mags.append(4.5)
print(mags)

event = {"event_id": "2016000291", "ml": 5.8, "depth_km": 15.0}
event["region"] = "Gyeongju"
print(event["ml"], list(event.keys()))

A **loop** (`for ... in ...:`) runs the indented lines once for every item; `if` runs lines only when a
test is `True`.

In [ ]:
n_big = 0
for m in mags:
    if m >= 5.0:
        n_big += 1
print(f"{n_big} of {len(mags)} magnitudes are 5.0 or larger")

A **function** (glossary `function`) is a named block of code that takes inputs (arguments) and returns an
output. Its **docstring**, the text in triple quotes right under the `def` line, says what goes in and
what comes out, with units; `help()` prints it. Small functions with a docstring each are one of the "good
enough practices" of Wilson et al. (2017). An **f-string** (`f"..."`) puts values into text, and a
format code after a colon says how: `{x:.2f}` two decimals, `{n:03d}` a whole number padded with zeros to 3
digits, `{s:>6}` right-aligned in 6 characters.

In [ ]:
def ml_label(magnitude, decimals=1):
    """Short text label for a local magnitude.

    magnitude : float, local magnitude ML
    decimals  : int, number of decimal places (default 1)
    returns   : str, for example 'ML 5.8'
    """
    return f"ML {magnitude:.{decimals}f}"


print(ml_label(5.8), "|", ml_label(2.25, decimals=2))
print(f"[{7:03d}] [{3.14159:.2f}] [{'KS':>6}]")
help(ml_label)

When Python stops with an error, it prints a **traceback**: the chain of function calls that led to the
error. Read it **from the bottom up**. The last line names the error type and gives a message; the lines
above show, call by call, where Python was, ending at the line that failed. The cell below makes an error on
purpose and prints the traceback without stopping the notebook (in your own work you will usually see the
traceback directly).

In [ ]:
def mean_magnitude(values):
    """Average of a list of magnitudes."""
    return sum(values) / len(values)


def describe(values):
    """One line of text about a list of magnitudes."""
    return f"mean {mean_magnitude(values):.2f} from {len(values)} events"


print(describe(mags))
try:
    print(describe([]))   # an empty list: what goes wrong, and where?
except ZeroDivisionError:
    traceback.print_exc()

The last line says `ZeroDivisionError: division by zero`. Just above it, the traceback points at
`return sum(values) / len(values)` inside `mean_magnitude`, which was called by `describe`, which was called
from this notebook cell. So `len(values)` was 0: the list was empty. **Try it:** in a new cell, run
`describe(["5.8"])` and read its traceback the same way.

## Part 2 · NumPy arrays and vectorization (about 45 min, then task 1)

A NumPy **array** is a block of numbers of one type, with fast math on all items at once. **Vectorization**
(glossary `vectorization`) means doing a calculation on a whole array at once instead of looping over the
items one by one. When an array meets a single number, NumPy uses that number for every item
(**broadcasting**). A comparison gives a **boolean array** (one `True` or `False` per item), and `sum()`
counts the `True` values. Harris et al. (2020) describe NumPy.

In [ ]:
depths = np.array([15.0, 12.0, 8.5, 14.0])
print(depths.dtype, depths.shape)
print(depths * 1000.0)             # kilometers to meters, for every item at once
print(depths > 10.0)               # a boolean array
print((depths > 10.0).sum(), "depths are deeper than 10 km")

NumPy's `sin`, `cos` and `arcsin` work in **radians** (a full circle is 2π radians, or 360 degrees).
`np.radians` converts degrees to radians and `np.degrees` converts back. `np.clip` keeps values inside a range.

In [ ]:
angles_deg = np.array([0.0, 30.0, 90.0])
print(np.sin(angles_deg))               # wrong on purpose: 30 is read as 30 radians
print(np.sin(np.radians(angles_deg)))   # right: 0, 0.5, 1
print(np.degrees(np.arcsin(0.5)))       # and back to degrees: 30
print(np.sqrt(np.array([0.0, 0.25, 1.0])))
print(np.clip(np.array([-0.1, 0.5, 1.0000000000000002]), 0.0, 1.0))   # rounding can leave a hair above 1

Why vectorize? Shorter code, and speed. Compare a Python loop with one array operation:

In [ ]:
rng = np.random.default_rng(DEMO_SEED)   # a random generator fixed by a random seed
x = rng.uniform(0.0, 10.0, N_DEMO)
t0 = time.perf_counter()
y_loop = [value * 2.0 for value in x]
t1 = time.perf_counter()
y_vec = x * 2.0
t2 = time.perf_counter()
print(f"loop {t1 - t0:.3f} s, vectorized {t2 - t1:.4f} s, same result: {np.allclose(y_loop, y_vec)}")

The **great-circle distance** (glossary `great_circle_distance`) is the shortest distance between two points
measured along the surface of the Earth. On a sphere it is the radius times the **central angle**, the angle
between the two points seen from the Earth's center. We need it for the distance from an epicenter to a
station or to another epicenter. Pythagoras on degrees does not work: one degree of longitude is about 111 km
at the equator but only about 90 km in Korea, and less still toward the poles.

### M01-T1 · Great-circle distance · core · about 90 min

**Goal:** write `haversine_km(lat1, lon1, lat2, lon2)`: the great-circle distance in kilometers between
points on a spherical Earth, for many pairs of points at once.

**Given:** the haversine formula. With latitudes $\varphi$ and longitudes $\lambda$ **in radians**,
$\Delta\varphi = \varphi_2 - \varphi_1$ and $\Delta\lambda = \lambda_2 - \lambda_1$:

$$a = \sin^2\!\left(\frac{\Delta\varphi}{2}\right) + \cos\varphi_1\,\cos\varphi_2\,
  \sin^2\!\left(\frac{\Delta\lambda}{2}\right), \qquad c = 2\arcsin\sqrt{a}, \qquad d = R\,c,$$

where $c$ is the central angle and $R$ = 6371.0 km (we treat the Earth as a sphere with this radius).

**Contract:**
- Inputs: latitudes and longitudes in decimal degrees (north and east positive), as plain numbers or as numpy
  arrays of the same shape. One point may be two plain numbers while the other points are arrays
  (broadcasting).
- Output: distances in kilometers, one per pair: a numpy array of the input shape, or one number for one pair.
- NumPy only, no Python loop. A ready-made distance function (pyproj, geopy, obspy.geodetics, scikit-learn,
  the haversine package) is not allowed: the check refuses it.
- Example: `haversine_km(0, 0, 0, 1)` is about 111.195 km (one degree along the equator).

**Done when** `st.check("M01-T1", func=haversine_km)` passes: 1000 fresh pairs (in Korea, around the globe,
and pairs only meters apart) agree with an independent library (ObsPy; Krischer et al. 2015) to 1 part in a
million, and two extra cases (one pair of plain numbers; one point against many) give results of the right
shape.

**Stuck?** `st.hint("M01-T1", 1)`

In [ ]:
# Before you code: what distance do you expect for one degree of latitude, and for the same point twice?
st.predict("M01-T1", """

""")

In [ ]:
EARTH_RADIUS_KM = 6371.0   # sphere radius fixed by the Contract


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance on a sphere of radius EARTH_RADIUS_KM.

    lat1, lon1 : float or numpy array, first point(s) in decimal degrees (north and east positive)
    lat2, lon2 : float or numpy array, second point(s) in decimal degrees
    returns    : distance(s) in kilometers, one per pair (numpy array, or one number for one pair)
    """
    # Step 1: convert the angles you need from degrees to radians (numpy works in radians).
    # Step 2: compute the term a of the Given formula, for all pairs at once.
    # Step 3: turn a into the central angle, then multiply by the radius.
    # YOUR CODE HERE
    raise NotImplementedError("M01-T1: write your code here")

In [ ]:
st.check("M01-T1", func=haversine_km)

**Explain (3-5 sentences):** Why must the angles be in radians before `np.sin`? What did vectorization save
you here? Use your function for the distance between the 2016 Gyeongju mainshock (35.76 N, 129.19 E) and the
2017 Pohang mainshock (36.11 N, 129.37 E), and say whether the value looks right on a map of Korea.

## Part 3 · Dates and time zones (about 45 min, then task 2)

The **origin time** (glossary `origin_time`) of an earthquake is the time when the rupture starts at the
hypocenter. Catalogs and **seismograms** (glossary `seismogram`: a record of ground motion against time at one
station) give it in **UTC** (glossary `utc`: Coordinated Universal Time, the reference clock for all times in
this tutorial). The KMA announces earthquakes in **KST** (glossary `kst`:
Korea Standard Time), and **KST = UTC + 9 h, always**. The catalog file that comes with the course stores UTC:
the `Z` at the end of `2016-09-12T11:32:54Z` means UTC, so the Gyeongju mainshock struck at 20:32:54 KST.

A **time zone** (glossary `time_zone`) is a rule that turns a moment into the clock reading of a place. A
**fixed offset** such as KST = UTC + 9 h is the simplest rule; Python's `zoneinfo` module also knows named time
zones from a time-zone database.

Python's `datetime` objects come in two kinds: **naive** (no time zone: only a clock reading) and **aware**
(with a time zone: a real moment). A `timedelta` is a length of time; adding one to a datetime does the
calendar arithmetic for you. `strftime` writes a datetime as text with format codes: `%Y` (4-digit year),
`%m` (month, 2 digits), `%d` (day), `%H` (hour 00-23), `%M` (minute), `%S` (second).

In [ ]:
t = dt.datetime(2019, 7, 6, 3, 19, 53, tzinfo=dt.timezone.utc)   # an aware moment in UTC (2019 Ridgecrest)
print(t.isoformat())
print(t + dt.timedelta(hours=36))          # 36 hours later: the date changes by itself
print(t - dt.timedelta(days=7))            # a week earlier
print(t.strftime("%d/%m/%Y at %H.%M"))     # the same moment as text, in a format of our choice

In [ ]:
ist = dt.timezone(dt.timedelta(hours=5, minutes=30))   # a fixed offset: India Standard Time, UTC + 5:30
print(t.astimezone(ist))                                # the same moment on an Indian clock
naive = dt.datetime(2019, 7, 6, 3, 19, 53)
print(naive.tzinfo, "| aware:", t.tzinfo)               # None means naive: Python does not know the zone
print(dt.datetime(2024, 2, 28, 23, 0) + dt.timedelta(hours=2))   # leap years are handled too

### M01-T2 · KST clock reading to a UTC event ID · core · about 60 min

**Goal:** write `kst_to_utc_id(year, month, day, hour, minute, second)`: turn a KST clock reading into a UTC
time ID, the text `'YYYYMMDDHHMMSS'` that our data files and tables use as a key.

**Given:** KST = UTC + 9 h, always (a fixed offset).

**Contract:**
- Inputs: six whole numbers (`int`): a date and a time of day in KST, second 0-59.
- Output: a `str` of 14 digits, the same moment in UTC as `YYYYMMDDHHMMSS`, zero-padded, no separators.
- Going back 9 hours can change the day, the month and the year (and February has 29 days in leap years).
- Example: the 2016 Gyeongju mainshock struck at 2016-09-12 20:32:54 KST, so
  `kst_to_utc_id(2016, 9, 12, 20, 32, 54)` returns `'20160912113254'`.

**Done when** `st.check("M01-T2", func=kst_to_utc_id)` passes on 200 fresh clock readings from 1978 to 2030
(with midnight, month, year and leap-day changes).

**Stuck?** `st.hint("M01-T2", 1)`

In [ ]:
# Before you code: for which KST readings is the UTC date not the KST date, and what else can change then?
st.predict("M01-T2", """

""")

In [ ]:
def kst_to_utc_id(year, month, day, hour, minute, second):
    """UTC time ID of a KST clock reading.

    year, month, day, hour, minute, second : int, a date and time of day in KST (UTC + 9 h)
    returns : str, the same moment in UTC written as 'YYYYMMDDHHMMSS' (14 digits)
    """
    # Step 1: make one datetime from the six numbers.
    # Step 2: go from KST to UTC with a fixed offset.
    # Step 3: write the result as 14 digits.
    # YOUR CODE HERE
    raise NotImplementedError("M01-T2: write your code here")

In [ ]:
st.check("M01-T2", func=kst_to_utc_id)

**Explain (3-5 sentences):** Which kinds of clock readings change the date when you go back 9 hours, and why
did the datetime object get them right without extra code? Why is it safer to store UTC in files and to
convert to KST only when you show a time to people?

## Part 4 · pandas: tables of earthquakes (about 60 min, then tasks 3 and 4)

A **DataFrame** (glossary `dataframe`) is a pandas table with named columns, like a spreadsheet inside
Python. One column is a **Series**; the row labels form the **index**. pandas (McKinney 2010) reads files with
`pd.read_csv`, selects columns by name (`df["ml"]`), selects rows with a boolean array (**boolean filtering**:
`df[df["ml"] >= 3]`), orders rows with `sort_values`, and summarizes groups of rows with `groupby`. We
practice on the station list that comes with the course.

In [ ]:
stations = st.data.stations()
print(stations.shape, "(rows, columns)")
print(stations.dtypes)
stations.head(N_SHOW)

In [ ]:
ks = stations[stations["network"] == "KS"]   # boolean filtering
print(len(ks), "KS stations")
north_kg = stations[(stations["network"] == "KG") & (stations["latitude"] > 37.0)]   # & means and
north_kg.sort_values("latitude", ascending=False).head(N_SHOW)   # sort_values orders the rows

In [ ]:
print(stations.groupby("network").size())   # how many stations each network has
print(stations.groupby("network")["latitude"].agg(["min", "max"]).round(2))

**Blank fields.** When a field of a CSV file is blank, `pd.read_csv` puts **NaN** there ("not a number",
pandas' marker for a missing value) and keeps the column as floats; `isna()` finds them. Never write 0 for a
missing depth: 0 km is a real depth.

**Byte order mark.** Some programs (Excel, for example) write a **byte order mark** (BOM) at the very start
of a UTF-8 text file: three invisible bytes, `EF BB BF`. pandas skips it, but Python's own `open()` with the
plain `"utf-8"` encoding keeps it as one invisible character, glued to the first column name; Python shows it
as the escape `'\ufeff'`, so the first name prints as `'\ufeffstation'`. The encoding `"utf-8-sig"` removes it.

In [ ]:
import csv
import io
import tempfile

small = pd.read_csv(io.StringIO("station,elevation_m\nSTA1,120\nSTA2,\n"))
print(small)
print(small["elevation_m"].isna().tolist(), small["elevation_m"].dtype)

with tempfile.TemporaryDirectory() as folder:
    demo = Path(folder) / "with_bom.csv"
    demo.write_text("station,elevation_m\nSTA1,120\n", encoding="utf-8-sig")   # writes a BOM first
    print(demo.read_bytes()[:3])                    # b'\xef\xbb\xbf' is the BOM
    with open(demo, encoding="utf-8") as fh:
        print(next(csv.reader(fh)))                 # the first name is '\ufeffstation'
    with open(demo, encoding="utf-8-sig") as fh:
        print(next(csv.reader(fh)))                 # clean: 'station'
    print(list(pd.read_csv(demo).columns))          # pandas copes on its own

**Times in pandas.** `pd.to_datetime` turns text into times; with `utc=True` they are aware and in UTC. It
also builds times from a table whose columns are the parts of a date. The `.dt` accessor works on a whole
column: `.dt.year`, `.dt.strftime(...)`, `.dt.tz_convert(zone)` (show the same moments on another clock) and
`.dt.tz_localize(zone)` (say which zone naive clock readings belong to, without shifting them). A
`pd.Timedelta(hours=...)` shifts a whole column.

In [ ]:
times = pd.to_datetime(pd.Series(["2016-09-12T10:44:32Z", "2016-09-12T11:32:54Z"]), utc=True)
print(times.dt.tz, "|", times.dt.strftime("%H:%M").tolist())
print(times.dt.tz_convert(ist).tolist())                          # the same moments on an Indian clock
clock = pd.Series(pd.to_datetime(["2020-03-01 00:30", "2020-12-31 23:30"]))
print(clock.dt.tz, "->", clock.dt.tz_localize("UTC").tolist())   # naive -> aware, no shift
print((clock + pd.Timedelta(hours=2)).tolist())                  # shift a whole column at once
parts = pd.DataFrame({"year": [2020], "month": [2], "day": [29], "hour": [6], "minute": [0], "second": [0]})
print(pd.to_datetime(parts).tolist())                            # times from columns of date parts

**The KMA catalog.** `st.data.kma_catalog()` loads the KMA catalog that comes with the course. It is used under
**KOGL Type 1** (glossary `kogl`: the Korea Open Government License, which lets anyone reuse public data if
they name the source); the attribution text is in `cat.attrs["source"]`. `origin_utc` is UTC; `origin_kst` shows the same moments with
the fixed +09:00 offset; `depth_km` is NaN where KMA gives no depth.

In [ ]:
cat = st.data.kma_catalog()
print(len(cat), "events from", cat["origin_utc"].min(), "to", cat["origin_utc"].max())
print(cat.attrs["source"].splitlines()[0])
cat.tail(N_SHOW)

In [ ]:
print(cat["depth_km"].isna().sum(), "events have no depth")
big = cat[cat["ml"] >= MIN_ML]
per_year = big.groupby(big["origin_utc"].dt.year).size()
print(f"events with ML >= {MIN_ML}, per year (last 10 years):")
print(per_year.tail(10))

### M01-T3 · Load a colleague's catalog file · core · about 120 min

**Goal:** write `load_catalog(path, tz)`: read a colleague's catalog file into a clean DataFrame.

**Given:** CSV files with the header `Year,Month,Day,Hour,Minute,Second,Latitude,Longitude,Depth,Magnitude`.
But some files list `Magnitude` before `Depth`, some start with a byte order mark, and `Depth` is blank when
it is not known. The six time columns are KST in some files and UTC in others: the file does not say; the
caller does, with `tz`.

**Contract:**
- Inputs: `path` (str or `pathlib.Path`) of the CSV file; `tz`, either `"KST"` or `"UTC"`: the clock of the
  six time columns.
- Output: a pandas DataFrame with exactly the columns `origin_utc, latitude, longitude, depth_km, ml`, in this
  order, one row per data line, in the file's order:
  - `origin_utc`: time-zone-aware times in UTC; KST readings move back 9 hours (fixed offset);
  - `latitude`, `longitude` (decimal degrees), `depth_km` (kilometers, NaN when blank) and `ml`: floats.
- Select columns by name, never by position.

**Done when** `st.check("M01-T3", func=load_catalog)` passes: the check writes 5 fresh files of all these
kinds into a temporary folder and compares your tables with what it wrote.

**Stuck?** `st.hint("M01-T3", 1)`

In [ ]:
# Before you code: which quirks of these files will pandas handle by itself, and which will need your own code?
st.predict("M01-T3", """

""")

In [ ]:
def load_catalog(path, tz):
    """Read a catalog CSV with separate time columns into a clean table.

    path    : str or pathlib.Path, a CSV file with the columns Year, Month, Day, Hour, Minute, Second,
              Latitude, Longitude, Depth, Magnitude (Depth and Magnitude in either order; the file may start
              with a byte order mark; Depth may be blank)
    tz      : "KST" or "UTC", the clock of the six time columns
    returns : pandas DataFrame with the columns origin_utc (aware, UTC), latitude, longitude,
              depth_km (NaN when blank), ml; one row per data line, in the file's order
    """
    # Step 1: read the file; later, pick columns by their names (never by position).
    # Step 2: build times from the six time columns; KST moves back 9 hours; declare the times UTC.
    # Step 3: put the five output columns together, in the Contract's order.
    # YOUR CODE HERE
    raise NotImplementedError("M01-T3: write your code here")

In [ ]:
st.check("M01-T3", func=load_catalog)

**Explain (3-5 sentences):** Which quirks of the files (KST times, Magnitude before Depth, the BOM, blank
depths) would have broken a quick first version of your function, and how did you find out? Why is NaN better
than 0 for a missing depth? What would you ask the colleague who sends you the next file?

### M01-T4 · Events within a radius, earliest first · core · about 60 min

**Goal:** write `events_within(df, lat0, lon0, r_km)`: the events of a catalog whose epicenter lies within
`r_km` of a point, earliest first.

**Given:** your `haversine_km` from M01-T1, and a catalog DataFrame with at least the columns `origin_utc`,
`latitude` and `longitude`, like `st.data.kma_catalog()`.

**Contract:**
- Inputs: `df` (DataFrame), `lat0`, `lon0` (decimal degrees), `r_km` (kilometers).
- Output: a DataFrame with the rows of `df` whose great-circle distance from (`lat0`, `lon0`) is at most
  `r_km` (distance <= `r_km`), with all of `df`'s columns, sorted by `origin_utc` from earliest to latest.
  When no event is close enough: an empty DataFrame with the same columns.
- Do not change `df` (no new column, no sorting in place).

**Done when** `st.check("M01-T4", func=events_within)` passes on 6 fresh made-up catalogs. Events within
0.01 km of the circle are left out of the test, so rounding cannot decide a case.

**Stuck?** `st.hint("M01-T4", 1)`

In [ ]:
# Before you code: about how many catalog events lie within EXAMPLE_RADIUS_KM of Gyeongju, and from which years?
st.predict("M01-T4", """

""")

In [ ]:
def events_within(df, lat0, lon0, r_km):
    """Events whose epicenter lies within r_km of the point (lat0, lon0), earliest first.

    df      : pandas DataFrame with at least the columns origin_utc, latitude, longitude (degrees)
    lat0    : float, latitude of the center in decimal degrees
    lon0    : float, longitude of the center in decimal degrees
    r_km    : float, radius in kilometers (an event exactly at r_km counts as inside)
    returns : pandas DataFrame, the rows of df within the radius with all of df's columns,
              sorted by origin_utc from earliest to latest; df itself is not changed
    """
    # Step 1: the distance from the center to every event, with your distance function, in one call.
    # Step 2: keep the rows that are close enough, then order them by time.
    # YOUR CODE HERE
    raise NotImplementedError("M01-T4: write your code here")

In [ ]:
st.check("M01-T4", func=events_within)

In [ ]:
near = events_within(cat, EXAMPLE_LAT, EXAMPLE_LON, EXAMPLE_RADIUS_KM)
print(f"{len(near)} catalog events within {EXAMPLE_RADIUS_KM} km of ({EXAMPLE_LAT}, {EXAMPLE_LON})")
print(near.groupby(near["origin_utc"].dt.year).size().tail(N_SHOW))
near.nlargest(N_SHOW, "ml")[["event_id", "origin_kst", "latitude", "longitude", "depth_km", "ml"]]

**Your selection becomes a picture.** Every dot is a row returned by your checked function. The left panel
shows when those events occurred and their sizes; the right shows their epicenters. A dense patch of dots
may reflect an earthquake sequence, detection changes, or both. M04–M05 develop the tests needed to tell.

**Look:** do the largest events and the most numerous events occur at the same times? Does choosing a radius
also choose a depth range? Name the information these two views cannot establish by themselves.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), constrained_layout=True)
axes[0].scatter(near["origin_utc"], near["ml"], s=9, alpha=0.45, color="tab:blue")
axes[0].set(xlabel="Origin time (UTC)", ylabel="Local magnitude ML", title="Events returned by your function")
axes[0].tick_params(axis="x", rotation=35)
points = axes[1].scatter(near["longitude"], near["latitude"], c=near["ml"], s=12, alpha=0.65, cmap="viridis")
axes[1].plot(EXAMPLE_LON, EXAMPLE_LAT, "r*", ms=12, label="Selection center")
axes[1].set(xlabel="Longitude (°E)", ylabel="Latitude (°N)", title=f"Epicenters within {EXAMPLE_RADIUS_KM:g} km")
axes[1].set_aspect(1 / np.cos(np.radians(EXAMPLE_LAT)))
axes[1].legend(fontsize=8)
fig.colorbar(points, ax=axes[1], label="Local magnitude ML")
show_figure(fig, "M01_selected_catalog.png")

**Explain (3-5 sentences):** Look at the output above (you may change the center and the radius in the
parameters cell). How many events are inside, and which are the largest? How did the result compare with
your prediction? Why does the check leave out events within 0.01 km of the circle?

## Part 5 · A one-hour mystery in old records (about 60 min)

A colleague made KST labels for every event in the catalog, to match the events with a Korean-language report
that lists KST times. The cell runs without an error and the labels look fine. Can they be trusted? To find
out, compare the colleague's way of converting the times with the fixed offset of task 2.

### M01-T5 · A one-hour mystery in old records (debug the trap) · core · about 60 min

**Goal:** write `kst_mismatch_summary(df)`: convert the same UTC moments to KST in two ways, count the events
whose two KST clock readings differ, and list the months of those events.

**Given:** the colleague's cell below. It converts the catalog's UTC times to KST with the `'Asia/Seoul'` time
zone (`ZoneInfo("Asia/Seoul")` from Python's `zoneinfo` module) and writes the labels into the column
`kst_label`. Run it, but do not edit it. The KMA catalog and the Korean report use KST = UTC + 9 h, a fixed
offset (task 2).

**Contract:**
- Input: `df`, a DataFrame with at least the column `origin_utc` (time-zone-aware times in UTC), like
  `st.data.kma_catalog()`. Use only this column: the made-up catalogs of the check have no `origin_kst`.
- Output: a dict with exactly two keys:
  - `"n_wrong"`: an `int`, the number of rows whose KST clock reading (date and time of day, to the second)
    is not the same in the two conversions: the fixed offset KST = UTC + 9 h, and the `'Asia/Seoul'` zone;
  - `"months"`: a sorted list of the months of those rows, written `'YYYY-MM'` from the fixed-offset KST
    reading, each month once (an empty list when no row differs).
- Do not change `df` (no new column).
- Example: a table in which every row agrees gives `{"n_wrong": 0, "months": []}`.

**Done when** `st.check("M01-T5", func=kst_mismatch_summary)` passes on 6 fresh made-up catalogs (their
answers are known from how they were built) and on the shipped KMA catalog (the check recomputes its answer
and never prints it).

**Stuck?** `st.hint("M01-T5", 1)`

In [ ]:
# The colleague's cell. Run it; do not edit it.
from zoneinfo import ZoneInfo

catalog = st.data.kma_catalog()
seoul = ZoneInfo("Asia/Seoul")
catalog["kst_seoul"] = catalog["origin_utc"].dt.tz_convert(seoul)
catalog["kst_label"] = catalog["kst_seoul"].dt.strftime("%Y-%m-%d %H:%M:%S")
catalog[["event_id", "origin_utc", "kst_label"]].tail(3)

In [ ]:
# Before you code: if some of the colleague's labels are wrong, which events do you expect, and how far off?
st.predict("M01-T5", """

""")

In [ ]:
def kst_mismatch_summary(df):
    """Events whose KST clock reading differs between a fixed +9 h offset and the 'Asia/Seoul' time zone.

    df      : pandas DataFrame with at least the column origin_utc (time-zone-aware, UTC)
    returns : dict with "n_wrong" (int, the number of rows whose two KST clock readings differ) and
              "months" (sorted list of the distinct 'YYYY-MM' of those rows, from the fixed-offset reading);
              df itself is not changed
    """
    # Step 1: convert origin_utc to KST in the two ways.
    # Step 2: find the rows whose two clock readings differ, and count them.
    # Step 3: collect the months of those rows (fixed-offset KST), each once, sorted.
    # YOUR CODE HERE
    raise NotImplementedError("M01-T5: write your code here")

In [ ]:
st.check("M01-T5", func=kst_mismatch_summary)

In [ ]:
summary = kst_mismatch_summary(catalog)
print(summary)
in_those_months = catalog["origin_kst"].dt.strftime("%Y-%m").isin(summary["months"])
catalog.loc[in_those_months, ["event_id", "origin_kst", "kst_seoul", "kst_label"]]

**Explain (3-5 sentences):** Look at the table above. Which labels are off, by how much and in which
direction? What does the `kst_seoul` column show for those events that a fixed offset never shows? If you
compare the two converted time stamps with `==`, they look equal: why? What would have gone wrong if these
labels had been used to match events with the Korean report, or to cut seismograms around the origin times?

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M01")

### Going further (optional, about 60 min, not checked)

1. With `groupby`, count the events with ML >= `MIN_ML` per year. Which year has the most? Why do the early
   years list fewer small events? (Think about the number of stations, not about the Earth.)
2. Set `EXAMPLE_LAT`, `EXAMPLE_LON` to the 2017 Pohang mainshock (36.11, 129.37) and `EXAMPLE_RADIUS_KM` to 10
   in the parameters cell, run the notebook again, and list the five largest events near Pohang.
3. Time your `haversine_km` on one million random pairs and compare it with a Python loop over the `math`
   module's functions. How many times faster is the vectorized version?

Next: M02 draws these catalogs as figures and maps.